In [121]:
import os, glob, re
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import tensorflow.keras.layers as L
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
import tensorflow.keras.regularizers as R
from tensorflow.keras.optimizers import Adam

In [122]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Paths

In [123]:
DRIVE_EXT = "/content/drive/My Drive"
IMAGE_DIR = f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images"
MASK_DIR = f"{DRIVE_EXT}/LabelStudioToMask_FINAL/masks"
sample_img = f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_1.png"
sample_mask = f"{DRIVE_EXT}/LabelStudioToMask_FINAL/masks/RVG_1_mask.png"

## Loading the images

In [124]:
def extract_rvg_id(path):
    filename = tf.strings.split(path, "/")[-1]
    noext = tf.strings.regex_replace(filename, ".png", "")
    rvg_id = tf.strings.regex_replace(noext, ".*(RVG_[0-9]+).*", r"\\1")
    return rvg_id

In [125]:
TARGET_SIZE = (790, 1100)
def load_image(image_path):
    img_obj = tf.io.read_file(image_path)
    image = tf.io.decode_png(img_obj, channels=1)
    image = tf.image.resize(image, TARGET_SIZE, method='bilinear')
    image = tf.pad(image, [[5,5],[2,2],[0,0]])   # pad H,W
    image = tf.cast(image, tf.float32) / 255.0 # Normalization
    return image

s = load_image(sample_img)
print("Sample image shape: ", s.shape)

Sample image shape:  (800, 1104, 1)


In [126]:
def load_mask(mask_path):
    """We have 4 classes."""
    mask = tf.io.decode_png(tf.io.read_file(mask_path), channels=1)
    mask = tf.image.resize(mask, TARGET_SIZE, method='nearest')
    mask = tf.pad(mask, [[5,5],[2,2],[0,0]])   # pad H,W
    mask = tf.cast(mask, tf.int32)
    mask = mask // 85   # 0→0, 85→1, 170→2, 255→3
    return mask

s1 = load_mask(sample_mask)
print("Shape of mask : ", s1.shape)
# print("Sample mask shape: ", s1.shape)

Shape of mask :  (800, 1104, 1)


In [127]:
def load_pair(image_path):
    """Extract name: RVG_1.png → RVG_1_mask.png"""
    filename = tf.strings.split(image_path, '/')[-1]   # RVG_1.png
    noext = tf.strings.regex_replace(filename, ".png", "")
    mask_name = noext + "_mask.png"
    mask_path = tf.strings.join([MASK_DIR, "/", mask_name])

    return load_image(image_path), load_mask(mask_path)


sample_pair = load_pair(sample_img)
print("Shapes of sample image: ", sample_pair[0].shape," And sample mask", sample_pair[1].shape)

Shapes of sample image:  (800, 1104, 1)  And sample mask (800, 1104, 1)


In [128]:
def extract_image_mask_patches(image, mask, patch_size=256, overlap=128):
    step = patch_size - overlap

    # ------------------------
    # FIX MASK SHAPE FIRST
    # Ensure mask is (H, W, 1), NOT (H, W) or (H, W, 1, 1)
    # ------------------------
    mask = tf.squeeze(mask)                 # Removes any extra dims
    mask = tf.expand_dims(mask, axis=-1)    # Shape: (H, W, 1)

    # ------------------------
    # IMAGE PATCHES
    # ------------------------
    img_patches = tf.image.extract_patches(
        images=tf.expand_dims(image, 0),
        sizes=[1, patch_size, patch_size, 1],
        strides=[1, step, step, 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )

    img_patches = tf.reshape(img_patches, [-1, patch_size, patch_size, 1])

    # ------------------------
    # MASK PATCHES
    # ------------------------
    mask_patches = tf.image.extract_patches(
        images=tf.expand_dims(mask, 0),
        sizes=[1, patch_size, patch_size, 1],
        strides=[1, step, step, 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )

    mask_patches = tf.reshape(mask_patches, [-1, patch_size, patch_size, 1])

    return img_patches, mask_patches


def patch_dataset(img, mask):
    img_p, mask_p = extract_image_mask_patches(img, mask)
    return tf.data.Dataset.from_tensor_slices((img_p, mask_p))


In [129]:
image_files = sorted([IMAGE_DIR + "/" + f for f in os.listdir(IMAGE_DIR)])
ds = tf.data.Dataset.from_tensor_slices(image_files)
ds = ds.map(load_pair, num_parallel_calls=4)

ds = ds.flat_map(patch_dataset) # Patching

#Splits
dataset_size = len(image_files)
train_size = int(0.7 * dataset_size)      # 61 for 88 images
val_size   = int(0.1 * dataset_size)     # 13 for 88 images
test_size  = dataset_size - train_size - val_size   # 14 for 88 images

BATCH_SIZE = 4

train_data = ds.take(train_size).batch(BATCH_SIZE)
val_data   = ds.skip(train_size).take(val_size).batch(BATCH_SIZE)
test_data  = ds.skip(train_size + val_size).take(test_size).batch(BATCH_SIZE)

print("Size of train data : ", train_data.cardinality().numpy())
print("Size of validation data : ", val_data.cardinality().numpy())
print("Size of test data : ", test_data.cardinality().numpy())
print("Total data size : ", train_data.cardinality().numpy() + val_data.cardinality().numpy() + test_data.cardinality().numpy())

Size of train data :  -2
Size of validation data :  -2
Size of test data :  -2
Total data size :  -6


## Defining the all blocks of the attention U-net block

In [130]:
def conv_block(inputs, num_filters):
    """This is the convolution block. There are two L in here. Takes input and number of filters as arguments"""
    x = L.Conv2D(num_filters, 3, padding="same")(inputs)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)

    x = L.Conv2D(num_filters, 3, padding="same")(x)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)

    return x

In [131]:
def attention_gate(x, g, filters):
    """ Attention gate to focus on important features."""
    theta_x = L.Conv2D(filters, 1)(x)
    phi_g   = L.Conv2D(filters, 1)(g)

    add = L.Add()([theta_x, phi_g])
    act = L.ReLU()(add)

    psi = L.Conv2D(1, 1, activation="sigmoid")(act)
    out = L.Multiply()([x, psi])   # attention-filtered skip

    return out


# Defining custom loss functions

In [132]:
def dice_loss(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)

    y_true = tf.one_hot(tf.cast(y_true, tf.int32), 4)
    y_pred = tf.nn.softmax(y_pred)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true + y_pred)

    return 1 - (2. * intersection + 1) / (union + 1)


def iou_metric(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)

    y_true = tf.one_hot(tf.cast(y_true, tf.int32), 4)
    y_pred = tf.nn.softmax(y_pred)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true + y_pred - (y_true * y_pred))

    return (intersection + 1) / (union + 1)

## Callbacks

In [133]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.8,
    patience=25,
    min_lr=1e-6,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="UNet_v5_bestmodel.h5",
    save_best_only=True,
    monitor="val_loss",
    mode="min"
)


callbacks = [reduce_lr, checkpoint]


# Defining the entirety of the U-net ++

![UNet++ Architecture](https://media.geeksforgeeks.org/wp-content/uploads/20230628132335/UNET.webp)


In [134]:
def attn_UNET_PP(input_shape, n_classes=4, init_filters=32): # Original init is 126 filters.
    inputs = L.Input(input_shape)

    # Number of filters
    f1 = init_filters
    f2 = f1 * 2
    f3 = f2 * 2
    f4 = f3 * 2
    f5 = f4 * 2

    # Encoder side
    x0_0 = conv_block(inputs, f1)
    p1   = L.MaxPool2D()(x0_0)

    x1_0 = conv_block(p1, f2)
    p2   = L.MaxPool2D()(x1_0)

    x2_0 = conv_block(p2, f3)
    p3   = L.MaxPool2D()(x2_0)

    x3_0 = conv_block(p3, f4)
    p4   = L.MaxPool2D()(x3_0)

    x4_0 = conv_block(p4, f5)

    # Decoder side + skip connections

    # Level 3
    g3 = L.UpSampling2D()(x4_0)
    a3 = attention_gate(x3_0, g3, f4)
    x3_1 = conv_block(L.Concatenate()([g3, a3]), f4)

    # Level 2
    g2 = L.UpSampling2D()(x3_1)
    a2_0 = attention_gate(x2_0, g2, f3)
    a2_1 = attention_gate(x2_0, g2, f3)

    x2_1 = conv_block(L.Concatenate()([g2, a2_0]), f3)
    x2_2 = conv_block(L.Concatenate()([x2_1, a2_1]), f3)

    # Level 1
    g1 = L.UpSampling2D()(x2_2)
    a1_0 = attention_gate(x1_0, g1, f2)
    a1_1 = attention_gate(x1_0, g1, f2)

    x1_1 = conv_block(L.Concatenate()([g1, a1_0]), f2)
    x1_2 = conv_block(L.Concatenate()([x1_1, a1_1]), f2)

    # Level 0
    g0 = L.UpSampling2D()(x1_2)
    a0_0 = attention_gate(x0_0, g0, f1)
    a0_1 = attention_gate(x0_0, g0, f1)

    x0_1 = conv_block(L.Concatenate()([g0, a0_0]), f1)
    x0_2 = conv_block(L.Concatenate()([x0_1, a0_1]), f1)

    # Deep supervision
    out1 = L.Conv2D(n_classes, 1, activation="softmax", name="ds1")(x0_1)

    out2 = L.Conv2D(n_classes, 1, activation="softmax", name="ds2")(x0_2)

    up3 = L.UpSampling2D(size=(2, 2), interpolation="bilinear")(x1_2)
    out3 = L.Conv2D(n_classes, 1, activation="softmax", name="ds3")(up3)

    up4 = L.UpSampling2D(size=(4, 4), interpolation="bilinear")(x2_2)
    out4 = L.Conv2D(n_classes, 1, activation="softmax", name="ds4")(up4)

    model = Model(inputs, [out1, out2, out3, out4])

    return model


In [135]:
input_shape = (None, None, 1)
model = attn_UNET_PP(input_shape, init_filters=32)
# loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

model.compile(
    optimizer="adam",
    loss={
        "ds1": "sparse_categorical_crossentropy",
        "ds2": "sparse_categorical_crossentropy",
        "ds3": "sparse_categorical_crossentropy",
        "ds4": "sparse_categorical_crossentropy"
    },
    metrics=[
        ["accuracy", iou_metric, dice_loss],
        ["accuracy", iou_metric, dice_loss],
        ["accuracy", iou_metric, dice_loss],
        ["accuracy", iou_metric, dice_loss],
    ]
)

print("model.output_shape:", model.output_shape)

model.summary()

model.output_shape: [(None, None, None, 4), (None, None, None, 4), (None, None, None, 4), (None, None, None, 4)]


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_180 (Conv2D) │ (None, None,      │        320 │ input_layer_4[0]… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_180[0][0]  │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_124 (ReLU)    │ (None, None,      │          0 │ batch_normalizat… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_181 (Conv2D) │ (None, None,      │      9,248 │ re_lu_124[0][0]   │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_181[0][0]  │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_125 (ReLU)    │ (None, None,      │          0 │ batch_normalizat… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_16    │ (None, None,      │          0 │ re_lu_125[0][0]   │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_182 (Conv2D) │ (None, None,      │     18,496 │ max_pooling2d_16… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_182[0][0]  │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_126 (ReLU)    │ (None, None,      │          0 │ batch_normalizat… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_183 (Conv2D) │ (None, None,      │     36,928 │ re_lu_126[0][0]   │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_183[0][0]  │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_127 (ReLU)    │ (None, None,      │          0 │ batch_normalizat… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_17    │ (None, None,      │          0 │ re_lu_127[0][0]   │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_184 (Conv2D) │ (None, None,      │     73,856 │ max_pooling2d_17… │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        512 │ conv2d_184[0][0]

 Total params: 8,769,463 (33.45 MB)

 Trainable params: 8,762,679 (33.43 MB)

 Non-trainable params: 6,784 (26.50 KB)

## Model training

In [136]:
def duplicate_mask(image, mask):
    return image, {
        "ds1": mask,
        "ds2": mask,
        "ds3": mask,
        "ds4": mask
    }

transformed_train_data = train_data.map(duplicate_mask)
transformed_val_data   = val_data.map(duplicate_mask)
transformed_test_data  = test_data.map(duplicate_mask)

In [137]:
for img, mask in transformed_train_data.take(1):
    print("IMAGE SHAPE  :", img.shape)
    print("MASK SHAPE   :", mask["ds1"].shape)  # or mask.shape if not dict


IMAGE SHAPE  : (4, 256, 256, 1)
MASK SHAPE   : (4, 256, 256, 1)


In [ ]:
history = model.fit(
    transformed_train_data,
    validation_data = transformed_val_data,
    epochs=200,
    callbacks=callbacks
)


Epoch 1/200
     46/Unknown 75s 221ms/step - ds1_accuracy: 0.5332 - ds1_dice_loss: 0.7185 - ds1_iou_metric: 0.1640 - ds1_loss: 1.3040 - ds2_accuracy: 0.5389 - ds2_dice_loss: 0.7129 - ds2_iou_metric: 0.1678 - ds2_loss: 1.2258 - ds3_accuracy: 0.4964 - ds3_dice_loss: 0.7182 - ds3_iou_metric: 0.1641 - ds3_loss: 1.2688 - ds4_accuracy: 0.5523 - ds4_dice_loss: 0.6984 - ds4_iou_metric: 0.1779 - ds4_loss: 1.2555 - loss: 5.0541

46/46 ━━━━━━━━━━━━━━━━━━━━ 90s 542ms/step - ds1_accuracy: 0.5336 - ds1_dice_loss: 0.7183 - ds1_iou_metric: 0.1641 - ds1_loss: 1.3021 - ds2_accuracy: 0.5394 - ds2_dice_loss: 0.7127 - ds2_iou_metric: 0.1679 - ds2_loss: 1.2242 - ds3_accuracy: 0.4973 - ds3_dice_loss: 0.7181 - ds3_iou_metric: 0.1642 - ds3_loss: 1.2671 - ds4_accuracy: 0.5522 - ds4_dice_loss: 0.6984 - ds4_iou_metric: 0.1779 - ds4_loss: 1.2543 - loss: 5.0477 - val_ds1_accuracy: 0.0812 - val_ds1_dice_loss: 0.8020 - val_ds1_iou_metric: 0.1099 - val_ds1_loss: 30.1994 - val_ds2_accuracy: 0.0959 - val_ds2_dice_loss: 0.7957 - val_ds2_iou_metric: 0.1139 - val_ds2_loss: 16.0938 - val_ds3_accuracy: 0.6718 - val_ds3_dice_loss: 0.6271 - val_ds3_iou_metric: 0.2299 - val_ds3_loss: 2.7743 - val_ds4_accuracy: 0.6539 - val_ds4_dice_loss: 0.6826 - val_ds4_iou_metric: 0.1888 - val_ds4_loss: 0.9261 - val_loss: 49.5274 - learning_rate: 0.0010
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step - ds1_accuracy: 0.5975 - ds1_dice_loss: 0.6986 - ds1

46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 236ms/step - ds1_accuracy: 0.5971 - ds1_dice_loss: 0.6986 - ds1_iou_metric: 0.1775 - ds1_loss: 1.0534 - ds2_accuracy: 0.5998 - ds2_dice_loss: 0.6955 - ds2_iou_metric: 0.1797 - ds2_loss: 1.0408 - ds3_accuracy: 0.5952 - ds3_dice_loss: 0.6961 - ds3_iou_metric: 0.1793 - ds3_loss: 1.0440 - ds4_accuracy: 0.5594 - ds4_dice_loss: 0.6942 - ds4_iou_metric: 0.1807 - ds4_loss: 1.0722 - loss: 4.2104 - val_ds1_accuracy: 0.3857 - val_ds1_dice_loss: 0.7345 - val_ds1_iou_metric: 0.1531 - val_ds1_loss: 1.2013 - val_ds2_accuracy: 0.4670 - val_ds2_dice_loss: 0.7330 - val_ds2_iou_metric: 0.1541 - val_ds2_loss: 1.1725 - val_ds3_accuracy: 0.3886 - val_ds3_dice_loss: 0.7309 - val_ds3_iou_metric: 0.1555 - val_ds3_loss: 1.1640 - val_ds4_accuracy: 0.5391 - val_ds4_dice_loss: 0.7229 - val_ds4_iou_metric: 0.1609 - val_ds4_loss: 1.0669 - val_loss: 4.6263 - learning_rate: 0.0010
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 226ms/step - ds1_accuracy: 0.6020 - ds1_dice_loss: 0.6915 - ds1_i

46/46 ━━━━━━━━━━━━━━━━━━━━ 31s 677ms/step - ds1_accuracy: 0.6048 - ds1_dice_loss: 0.6881 - ds1_iou_metric: 0.1849 - ds1_loss: 0.9884 - ds2_accuracy: 0.6022 - ds2_dice_loss: 0.6880 - ds2_iou_metric: 0.1850 - ds2_loss: 0.9913 - ds3_accuracy: 0.6001 - ds3_dice_loss: 0.6849 - ds3_iou_metric: 0.1872 - ds3_loss: 0.9930 - ds4_accuracy: 0.5827 - ds4_dice_loss: 0.6860 - ds4_iou_metric: 0.1865 - ds4_loss: 0.9885 - loss: 3.9612 - val_ds1_accuracy: 0.6787 - val_ds1_dice_loss: 0.7104 - val_ds1_iou_metric: 0.1694 - val_ds1_loss: 0.9849 - val_ds2_accuracy: 0.6953 - val_ds2_dice_loss: 0.7125 - val_ds2_iou_metric: 0.1679 - val_ds2_loss: 0.9996 - val_ds3_accuracy: 0.6571 - val_ds3_dice_loss: 0.7051 - val_ds3_iou_metric: 0.1730 - val_ds3_loss: 0.9532 - val_ds4_accuracy: 0.6980 - val_ds4_dice_loss: 0.7126 - val_ds4_iou_metric: 0.1678 - val_ds4_loss: 1.0091 - val_loss: 4.0073 - learning_rate: 0.0010
Epoch 5/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 225ms/step - ds1_accuracy: 0.6045 - ds1_dice_loss: 0.6866 - ds1_i

46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 238ms/step - ds1_accuracy: 0.6011 - ds1_dice_loss: 0.6876 - ds1_iou_metric: 0.1853 - ds1_loss: 0.9820 - ds2_accuracy: 0.5982 - ds2_dice_loss: 0.6865 - ds2_iou_metric: 0.1861 - ds2_loss: 0.9755 - ds3_accuracy: 0.5805 - ds3_dice_loss: 0.6885 - ds3_iou_metric: 0.1847 - ds3_loss: 0.9845 - ds4_accuracy: 0.5640 - ds4_dice_loss: 0.6885 - ds4_iou_metric: 0.1846 - ds4_loss: 1.0141 - loss: 3.9561 - val_ds1_accuracy: 0.6510 - val_ds1_dice_loss: 0.7057 - val_ds1_iou_metric: 0.1726 - val_ds1_loss: 0.9764 - val_ds2_accuracy: 0.6923 - val_ds2_dice_loss: 0.7020 - val_ds2_iou_metric: 0.1751 - val_ds2_loss: 0.9552 - val_ds3_accuracy: 0.6566 - val_ds3_dice_loss: 0.7015 - val_ds3_iou_metric: 0.1755 - val_ds3_loss: 0.9280 - val_ds4_accuracy: 0.7053 - val_ds4_dice_loss: 0.6939 - val_ds4_iou_metric: 0.1808 - val_ds4_loss: 0.8896 - val_loss: 3.8107 - learning_rate: 0.0010
Epoch 7/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 229ms/step - ds1_accuracy: 0.6009 - ds1_dice_loss: 0.6861 - ds1_i

## Plotting loss curves

In [ ]:
his = history.history

plt.figure(figsize=(22, 12))

# -------------------------------------------------------
# Plot 1: Metrics for ds1
# -------------------------------------------------------
plt.subplot(2, 3, 1)
plt.plot(his["ds1_loss"], label="ds1_loss")
plt.plot(his["ds1_accuracy"], label="ds1_accuracy")
plt.plot(his["ds1_dice_loss"], label="ds1_dice_loss")
plt.plot(his["ds1_iou_metric"], label="ds1_iou_metric")
plt.title("Metrics for ds1")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.grid()

# -------------------------------------------------------
# Plot 2: Metrics for ds2
# -------------------------------------------------------
plt.subplot(2, 3, 2)
plt.plot(his["ds2_loss"], label="ds2_loss")
plt.plot(his["ds2_accuracy"], label="ds2_accuracy")
plt.plot(his["ds2_dice_loss"], label="ds2_dice_loss")
plt.plot(his["ds2_iou_metric"], label="ds2_iou_metric")
plt.title("Metrics for ds2")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.grid()

# -------------------------------------------------------
# Plot 3: Metrics for ds3
# -------------------------------------------------------
plt.subplot(2, 3, 3)
plt.plot(his["ds3_loss"], label="ds3_loss")
plt.plot(his["ds3_accuracy"], label="ds3_accuracy")
plt.plot(his["ds3_dice_loss"], label="ds3_dice_loss")
plt.plot(his["ds3_iou_metric"], label="ds3_iou_metric")
plt.title("Metrics for ds3")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.grid()

# -------------------------------------------------------
# Plot 4: Metrics for ds4
# -------------------------------------------------------
plt.subplot(2, 3, 4)
plt.plot(his["ds4_loss"], label="ds4_loss")
plt.plot(his["ds4_accuracy"], label="ds4_accuracy")
plt.plot(his["ds4_dice_loss"], label="ds4_dice_loss")
plt.plot(his["ds4_iou_metric"], label="ds4_iou_metric")
plt.title("Metrics for ds4")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.grid()

# -------------------------------------------------------
# Plot 5: All accuracies + val accuracies
# -------------------------------------------------------
plt.subplot(2, 3, 5)
for ds in ["ds1", "ds2", "ds3", "ds4"]:
    plt.plot(his[f"{ds}_accuracy"], label=f"{ds}_accuracy")
    plt.plot(his[f"val_{ds}_accuracy"], label=f"val_{ds}_accuracy")

plt.title("Accuracy + Val Accuracy (All Outputs)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()

# -------------------------------------------------------
# Plot 6: All losses + val losses
# -------------------------------------------------------
plt.subplot(2, 3, 6)
for ds in ["ds1", "ds2", "ds3", "ds4"]:
    plt.plot(his[f"{ds}_loss"], label=f"{ds}_loss")
    plt.plot(his[f"val_{ds}_loss"], label=f"val_{ds}_loss")

plt.title("Loss + Val Loss (All Outputs)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()


## Testing

In [ ]:
results = model.evaluate(transformed_test_data)
print("Test metrics:", results)


## Testing with 1 image

In [ ]:
TEST_IMAGE = f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_109.png"
TEST_MASK = f"{DRIVE_EXT}/LabelStudioToMask_FINAL/masks/RVG_109_mask.png"

In [ ]:
img = load_image(TEST_IMAGE)   # applies padding/resize etc.
img = tf.expand_dims(img, axis=0)  # shape: (1, H, W, 1)

preds = model.predict(img) #preds is a list

label_maps = []
for pred in preds:
  label_map = tf.argmax(pred, axis=-1)
  label_map = label_map[0]

  label_maps.append(label_map)


In [ ]:
gt_mask = load_mask(TEST_MASK)
gt_mask = tf.squeeze(gt_mask).numpy()


In [ ]:
import matplotlib.pyplot as plt
HEADS = 4
SIZE = HEADS + 2
plt.figure(figsize=(30,8))

plt.subplot(1,SIZE,1)
plt.title("Input Image")
plt.imshow(tf.squeeze(img[0]), cmap='gray')
plt.axis('off')

plt.subplot(1,SIZE,2)
plt.title("Ground Truth Mask")
plt.imshow(gt_mask, cmap='PuOr')
plt.axis('off')

plt.subplot(1,SIZE,3)
plt.title("Predicted Mask head 1")
plt.imshow(label_maps[0], cmap='PuOr')
plt.axis('off')

plt.subplot(1,SIZE,4)
plt.title("Predicted Mask head 2")
plt.imshow(label_maps[1], cmap='PuOr')
plt.axis('off')

plt.subplot(1,SIZE,5)
plt.title("Predicted Mask head 3")
plt.imshow(label_maps[2], cmap='PuOr')
plt.axis('off')

plt.subplot(1,SIZE,6)
plt.title("Predicted Mask head 4")
plt.imshow(label_maps[3], cmap='PuOr')
plt.axis('off')

plt.savefig('/kaggle/working.')
plt.show()


## Testing with 10 images

In [ ]:
TEST_IMAGES = [
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_101.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_102.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_103.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_104.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_105.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_106.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_107.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_108.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_109.png",
    f"{DRIVE_EXT}/LabelStudioToMask_FINAL/images/RVG_110.png",
]

In [ ]:
def predict_all_heads(model, image_path, mask_path):
    img = load_image(image_path)
    img_batch = tf.expand_dims(img, axis=0)

    preds = model.predict(img_batch)

    label_maps = [tf.argmax(p, axis=-1)[0] for p in preds]
    gt_mask = load_mask(mask_path)
    gt_mask = tf.squeeze(gt_mask)

    return img, gt_mask, label_maps

def plot_row(img, gt_mask, label_maps, row_idx, total_rows):
    cols = 6  # Input + GT + 4 heads

    plt.subplot(total_rows, cols, row_idx * cols + 1)
    plt.imshow(tf.squeeze(img), cmap="gray")
    plt.title("Image")
    plt.axis("off")

    plt.subplot(total_rows, cols, row_idx * cols + 2)
    plt.imshow(gt_mask, cmap="PuOr")
    plt.title("GT")
    plt.axis("off")

    for i in range(4):   # 4 heads
        plt.subplot(total_rows, cols, row_idx * cols + 3 + i)
        plt.imshow(label_maps[i], cmap="PuOr")
        plt.title(f"Head {i+1}")
        plt.axis("off")


In [ ]:
num_imgs = len(TEST_IMAGES)
cols = 6
plt.figure(figsize=(cols * 3, num_imgs * 3))

for idx, img_path in enumerate(TEST_IMAGES):
    base = img_path.split("/")[-1].replace(".png", "")
    mask_path = f"{DRIVE_EXT}/LabelStudioToMask_FINAL/masks/{base}_mask.png"

    # Predict
    img, gt_mask, label_maps = predict_all_heads(model, img_path, mask_path)

    # Plot in grid
    plot_row(img, gt_mask, label_maps, idx, num_imgs)

plt.tight_layout()
plt.show()
